In [1]:
import time
import pandas as pd
from datasets import load_dataset
import SlideScholarDB as ssdb
from pprint import pprint
import os

# load dataset

In [2]:
ds = load_dataset("kd13/bookcorpus-clean")

ds

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'sent_id', 'text'],
        num_rows: 33649142
    })
})

In [3]:
texts = ds['train'][:10000]['text'] # choose first 10000 sample
print(len(texts))
print(texts[0])

10000
i wish i had a better answer to that question .


# Save document to db

In [4]:
# Build chunks
# A chunk is a dictionary that form like this:
#
# {
#     'text': text used to embed,
#     'metadata': Can be anything, used to store origin data
# }

chunks = [{
    'text': t,
    'metadata': {'text': t}
} for t in texts]

pprint(chunks[0])

{'metadata': {'text': 'i wish i had a better answer to that question .'},
 'text': 'i wish i had a better answer to that question .'}


In [ ]:
# init database

db = ssdb.vdb(verbose=True) # set verbose parameter to see the log

db.add_documents(chunks)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[VDB.__init__] Model intfloat/e5-large-v2 loaded on mps.
[VDB.add_documents] Adding 10000 documents...
[VDB.add_documents] Embedding 10000 passages...
[VDB._embed_texts] Embedding 10000 texts with prefix 'passage: '...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

[VDB.add_documents] Building ivfflat index...
[VDB._build_index] Building index of type 'ivfflat' with 10000 embeddings...
[VDB.build_index] Training IVFFlat index...


# Search document from db

In [6]:
db.search("people's imagination")

[VDB.search] Searching for query: 'people's imagination' (top_k=5)...
[VDB._embed_texts] Embedding 1 texts with prefix 'query: '...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'score': 0.7851935625076294,
  'chunk': {'text': "when the story was over , she could see in the kids ' eyes that they had imagined every peal of thunder , felt the shuddering of the splintered wooden boat on the open sea .",
   'metadata': {'text': "when the story was over , she could see in the kids ' eyes that they had imagined every peal of thunder , felt the shuddering of the splintered wooden boat on the open sea ."}}},
 {'score': 0.778754711151123,
  'chunk': {'text': 'coroico , with its stunning scenery , picturesque small-town charm , thick with plantings of coffee beans and oranges ?',
   'metadata': {'text': 'coroico , with its stunning scenery , picturesque small-town charm , thick with plantings of coffee beans and oranges ?'}}},
 {'score': 0.7752187252044678,
  'chunk': {'text': 'groups of fair-skinned tourists , wearing khaki shorts with hiking sandals and socks , sat around the plaza laughing too loudly and munching pringles and snickers bars that the local stores kep

# save&load db

In [7]:
if not os.path.exists('example/'):
    os.mkdir('example/')

db.save_index('example/example_id.index', 'example/example_chunks.json')

[VDB.save_index] Saving index to example/example_id.index...
[VDB.save_index] Saving chunks to example/example_chunks.json...


In [9]:
new_db = ssdb.vdb()

new_db.load_index('example/example_id.index', 'example/example_chunks.json')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
new_db.search("people's imagination")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'score': 0.7851935625076294,
  'chunk': {'text': "when the story was over , she could see in the kids ' eyes that they had imagined every peal of thunder , felt the shuddering of the splintered wooden boat on the open sea .",
   'metadata': {'text': "when the story was over , she could see in the kids ' eyes that they had imagined every peal of thunder , felt the shuddering of the splintered wooden boat on the open sea ."}}},
 {'score': 0.778754711151123,
  'chunk': {'text': 'coroico , with its stunning scenery , picturesque small-town charm , thick with plantings of coffee beans and oranges ?',
   'metadata': {'text': 'coroico , with its stunning scenery , picturesque small-town charm , thick with plantings of coffee beans and oranges ?'}}},
 {'score': 0.7752187252044678,
  'chunk': {'text': 'groups of fair-skinned tourists , wearing khaki shorts with hiking sandals and socks , sat around the plaza laughing too loudly and munching pringles and snickers bars that the local stores kep

# embed lecture document chunks

In [11]:
chunk_path = "example/chunks.json"

db = ssdb.read_chunks(chunk_path)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

WARNING clustering 1431 points to 100 centroids: please provide at least 3900 training points


In [12]:
query = "Synchronous Gradient Descent"

db.search(query)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'score': 0.8509376049041748,
  'chunk': {'text': 'Convergence-Runtime Tradeoff in SGD\nVariants\n• Error-runtime trade-off for Sync and Async-SGD with same learning rate.\n• Async-SGD has faster decay with time but a higher error floor.\n\n[VISUAL CONTENT]:\nThe slide is teaching the concept of intelligent-run-time tradeoff in SGD (Stochastic Gradient Descent) learning. It shows a graph with two lines, one labeled "log loss" and the other labeled "asynchronous." The graph is described as a log-log plot, with the x-axis representing the number of iterations and the y-axis representing the log loss.\n\nThe slide also includes a flowchart that illustrates the tradeoff between synchronous and asynchronous learning. The flowchart shows the different levels of learning, including synchronous and asynchronous learning, and how they can be optimized for faster decay with time but with a higher error floor.',
   'metadata': {'name': 'Lecture 1',
    'lecture_num': 1.0,
    'slide': 84,
    's